### Load System Libraries

In [ ]:
import sys
import os

sys.path.append(os.path.abspath(os.path.join("..", "useful-python-scripts-eda")))
from data_profiler import DataProfiler
from distribution_analyzer import DistributionAnalyzer
from correlation_explorer import CorrelationExplorer
from outlier_suite import OutlierSuite
from missing_data_analyzer import MissingDataAnalyzer

import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

from relbench.datasets import get_dataset

### Load Datasets

In [ ]:
db = get_dataset("rel-hm", download=True).get_db(upto_test_timestamp=False)

In [ ]:
articles_df = db.table_dict["article"].df
customers_df = db.table_dict["customer"].df
transactions_df = db.table_dict["transactions"].df

### Generate master database

In [ ]:
customers_cols = [col for col in customers_df.columns if col not in ["postal_code", "fashion_news_frequency"]]
df = transactions_df.merge(customers_df[customers_cols], on="customer_id", how="left")

In [ ]:
# Some columns are clearly categorical, so we transform them to string datatype.
categorical_cols = ["customer_id", "article_id", "sales_channel_id", "FN", "Active"]
for col in categorical_cols:
    df[col] = df[col].astype(str)

In [ ]:
df.head(5)

### Comprehensive Data Profiler

In [ ]:
# Create profiler.
profiler = DataProfiler(df, high_cardinality_threshold=0.5)

In [ ]:
# Print summary to console.
profiler.print_summary()

In [ ]:
# Get detailed report.
profiler_report = profiler.generate_full_profile()
print(profiler_report["overview"])
print("--" * 20)
print(profiler_report["numeric_profiles"])
print("--" * 20)
print(profiler_report["categorical_profiles"])
print("--" * 20)
print(profiler_report["data_quality_issues"])

### Analyzing And Visualizing Distributions

In [ ]:
analyzer = DistributionAnalyzer(df)

In [ ]:
# Generate distribution report
analyzer_report = analyzer.generate_distribution_report()
print(analyzer_report)

In [ ]:
# Identify highly skewed columns
skewed = analyzer_report[abs(analyzer_report['skewness']) > 2]
print(f"Highly skewed columns: {skewed['column'].tolist()}")

In [ ]:
# Visualize distributions
analyzer.plot_numeric_distributions(max_cols=10)
analyzer.plot_boxplots()
analyzer.plot_qq_plots()
analyzer.plot_categorical_distributions()

### Correlation and Relationship Explorer

In [ ]:
explorer = CorrelationExplorer(df)

In [ ]:
# Find high correlations
high_corr = explorer.find_high_correlations(threshold=0.7, method='pearson')
print(high_corr)

In [ ]:
# Check for multicollinearity
vif = explorer.calculate_vif()
problematic = vif[vif['vif'] > 10]
print(f"Features with high multicollinearity:\n{problematic}")

In [ ]:
# Mutual information with target
if 'target' in df.columns:
    mi_scores = explorer.mutual_information_analysis('target')
    print(f"Top features:\n{mi_scores.head(10)}")

In [ ]:
# Visualize
explorer.plot_correlation_heatmap(method='pearson')
explorer.plot_correlation_comparison()
explorer.plot_scatter_matrix(max_cols=5)
explorer.plot_top_correlations(n_pairs=10)

### Outlier Detection and Analysis Suite

In [ ]:
suite = OutlierSuite(df)

In [ ]:
# Compare methods across all columns
summary = suite.compare_methods_all_columns()
print(summary)

In [ ]:
# Analyze specific column
col = "price"
suite.plot_outlier_comparison(col)

In [ ]:
# Detect multivariate outliers
iso_outliers = suite.detect_isolation_forest_outliers(contamination=0.1)
print(f"Found {iso_outliers.sum()} multivariate outliers")

feature1 = "price"
feature2 = "age"
suite.plot_multivariate_outliers([feature1, feature2])

In [ ]:
# Analyze outlier impact
col = "price"
impact = suite.analyze_outlier_impact(col)
print(impact)

### Missing Data Pattern Analyzer

In [ ]:
analyzer = MissingDataAnalyzer(df)

In [ ]:
# Generate full report
report = analyzer.generate_full_report()

print("Missing Value Summary:")
print(report['summary'])

print("\nMissingness Patterns (co-occurrence):")
print(report['patterns'])

print("\nMissingness Classifications:")
print(report['classifications'])

print("\nImputation Recommendations:")
print(report['recommendations'])

In [ ]:
# Visualize patterns
analyzer.plot_missing_bar()
analyzer.plot_missing_heatmap(max_cols=30)
analyzer.plot_missing_correlation()

In [ ]:
# Classify specific column
col = "age"
classification = analyzer.classify_missingness_type(col)
recommendation = analyzer.recommend_strategy(col)
print(f"Missingness type: {classification['missingness_type']}")
print(f"Recommendation: {recommendation}")